In [ ]:
!pip install gradio sentence-transformers scikit-learn joblib torch

import os
import re
import joblib
import numpy as np
import torch
import gradio as gr

from sentence_transformers import SentenceTransformer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.neighbors import KNeighborsClassifier

# 1. Load your saved KNN model from Drive
MODEL_PATH = "/content/drive/MyDrive/best_overall_model.pkl"
model = joblib.load(MODEL_PATH)

# 2. Decide GPU or CPU for embeddings
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")

# 3. Minimal text cleanup
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

# 4. Probability predictor
def predict_coverage_prob(text_input: str) -> np.ndarray:
    embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
    cleaned = clean_text(text_input)
    X_emb = embedder.encode([cleaned])
    
    coverage_probs = []
    for est in model.estimators_:
        if len(est.classes_) == 1:
            single_label_val = est.classes_[0]
            # If it's 1, prob=1. If it's 0, prob=0
            label_prob = 1.0 if single_label_val == 1 else 0.0
        else:
            probs = est.predict_proba(X_emb)  # shape (1,2)
            label_prob = probs[0, 1]         # prob of class=1
        coverage_probs.append(label_prob)

    return np.array(coverage_probs)  # shape (17,)

# 5. Format to show only SDGs > 0
def chatbot_sdg_coverage(user_input: str) -> str:
    coverage_arr = predict_coverage_prob(user_input)
    lines = []
    for i, val in enumerate(coverage_arr):
        pct = val * 100
        if pct > 0:
            lines.append(f"SDG {i+1}: {pct:.1f}%")
    if not lines:
        lines = ["No coverage predicted."]
    return "\n".join(lines)

# 6. Gradio UI
iface = gr.Interface(
    fn=chatbot_sdg_coverage,
    inputs="text",
    outputs="text",
    title="SDG Coverage Chatbot",
    description="Enter text describing a project or initiative. The model will display only non-zero SDGs coverage."
)
iface.launch()
